### Data Ingestion

In [2]:
# Document Structure
from langchain_core.documents import Document

doc = Document(
    page_content = "this is the main text content I am using to create RAG",
    metadata = {
        "source":"example.txt",
        "pages":1,
        "author":"Elsaka",
        "date_created":"2025-1-10" 
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Elsaka', 'date_created': '2025-1-10'}, page_content='this is the main text content I am using to create RAG')

In [3]:
# Create a simple text file 
import os 
os.makedirs("../data/text_files", exist_ok= True)

In [4]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath, content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)
        
print("Sample text files created!")


Sample text files created!


In [5]:
# Text Loader
# Upload a text file and convert its content to a Document within LangChain
from langchain_community.document_loaders import TextLoader

loader = TextLoader ("../data/text_files/python_intro.txt", encoding="utf-8")
document = loader.load()
print(document)

d:\Projects Ds\RAG_APP\.venv\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [6]:
# To download all text files within an entire folder
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/text_files",
    glob= "**/*.txt", 
    loader_cls= TextLoader,
    loader_kwargs= {'encoding':'utf-8'},
    show_progress= False
)
documents = dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n    \n    \n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the m

In [7]:
# # To download all PDF files within an entire folder
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob= "**/*.pdf", 
    loader_cls= PyMuPDFLoader,
    show_progress= False
)
pdf_doc = dir_loader.load()
pdf_doc

d:\Projects Ds\RAG_APP\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'producer': 'Pdftools SDK', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\Mastering RAG.pdf', 'file_path': '..\\data\\pdf\\Mastering RAG.pdf', 'total_pages': 194, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-09-27T00:48:54+00:00', 'trapped': '', 'modDate': 'D:20240927004854Z', 'creationDate': '', 'page': 0}, page_content='Mastering\nRAG\nA comprehensive guide for building\nenterprise-grade RAG systems'),
 Document(metadata={'producer': 'Pdftools SDK', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\Mastering RAG.pdf', 'file_path': '..\\data\\pdf\\Mastering RAG.pdf', 'total_pages': 194, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-09-27T00:48:54+00:00', 'trapped': '', 'modDate': 'D:20240927004854Z', 'creationDate': '', 'page': 1}, page_content='!"#$%&\'$()*\'")*+%,-.%/0)(123%.4#54%+-""4*%\n\'((0$"-647%"-%89:-69"%\'*7%3-0%$,\'22%+4";%\n.)<\

In [8]:
# Text Splitter
# Dividing long texts into smaller chunks for easier processing in AI models.
# Each chunk is 800 characters long with an overlap of 100 characters between chunks to maintain contextual meaning.
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", "!", "?", ",", " "]
)

chunks = text_splitter.split_documents(pdf_doc)
print(f"Split {len(pdf_doc)} documents into {len(chunks)} chunks.")

Split 194 documents into 678 chunks.


### Embeddings

In [10]:
import numpy as np 
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
class EmbeddingManager:
    """Handle Doc Embedding Gen using SentenceTransformer"""
    def __init__(self, model_name: str="all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model= None
        self._load_Model()
        
    def _load_Model(self):
        try:
            print(f"loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully: {self.model.get_sentence_embedding_dimension()}") # It prints the number of dimensions that the model uses in the embeddings.
        except Exception as e : 
            print(f"ERROR Laoding model {self.model_name}:{e}") 
            raise   
    
    def generate_embedding(self, texts: list[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        print (f"generated embedings for {len(texts)} texts...") 
        # It converts each text in the list into a vector  
        embedings = self.model.encode(texts, show_progress_bar = True)
        print (f"generated embed with shape :{embedings.shape}")
        return embedings
    
Embedd_manger = EmbeddingManager()
Embedd_manger

loading model: all-MiniLM-L6-v2
Model loaded successfully: 384


### VectorStore

In [12]:
import os
import uuid
import numpy as np
import chromadb
from typing import Any

class VectorStore:
    """Manage Document embeddings in a ChromaDB vector store"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name # The name of the collection where the data will be stored.
        self.persist_directory = persist_directory # The location where ChromaDB data will be stored.
        self.client = None
        self.collection = None
        self._initialize_store() # It calls another specific function to prepare the database.
       
    def _initialize_store(self):
        """Initialize Chroma client and create/get collection"""
        try:
            print(f"Initializing ChromaDB at {self.persist_directory} ...")
            os.makedirs(self.persist_directory, exist_ok=True)
            
            # We're creating this so that the data won't be lost when the program closes.
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # create or retrive collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF Document Embeddings for RAG"}
            )
            print(f"Vector store initialized successfully with collection: {self.collection_name}")
            print(f"Existing documents count: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise
        
    def add_documents(self, documents: list[Any], embeddings: np.ndarray):
        """Add documents and embeddings to the ChromaDB collection"""
        
        # This ensures that each document has its own unique embedding.
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to VectorStore...")
        
        # Prepare 4 lists for storing data.
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            metadata = dict(doc.metadata) if hasattr(doc, "metadata") else {}
            metadata["doc_index"] = i 
            metadata["content_len"] = len(getattr(doc, "page_content", str(doc)))
            metadatas.append(metadata)
            
            documents_text.append(getattr(doc, "page_content", str(doc)))
            embeddings_list.append(embedding.tolist())
            
        try:
            self.collection.add(
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text,
                ids=ids
            )
            print(f"Successfully added {len(documents)} documents.")
            print(f"Total documents now: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents: {e}")
            raise

vecstore = VectorStore()
vecstore


Initializing ChromaDB at ../data/vector_store ...
Vector store initialized successfully with collection: pdf_documents
Existing documents count: 4748


In [13]:
chunks

[Document(metadata={'producer': 'Pdftools SDK', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\Mastering RAG.pdf', 'file_path': '..\\data\\pdf\\Mastering RAG.pdf', 'total_pages': 194, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-09-27T00:48:54+00:00', 'trapped': '', 'modDate': 'D:20240927004854Z', 'creationDate': '', 'page': 0}, page_content='Mastering\nRAG\nA comprehensive guide for building\nenterprise-grade RAG systems'),
 Document(metadata={'producer': 'Pdftools SDK', 'creator': '', 'creationdate': '', 'source': '..\\data\\pdf\\Mastering RAG.pdf', 'file_path': '..\\data\\pdf\\Mastering RAG.pdf', 'total_pages': 194, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2024-09-27T00:48:54+00:00', 'trapped': '', 'modDate': 'D:20240927004854Z', 'creationDate': '', 'page': 1}, page_content='!"#$%&\'$()*\'")*+%,-.%/0)(123%.4#54%+-""4*%\n\'((0$"-647%"-%89:-69"%\'*7%3-0%$,\'22%+4";%\n.)<\

In [14]:
# Convert Text to embeddings
texts = [doc.page_content for doc in chunks]
# Generate Embeddings
embeddings = Embedd_manger.generate_embedding(texts)
# Store in VectorDB
vecstore.add_documents(chunks, embeddings)

generated embedings for 678 texts...


Batches: 100%|██████████| 22/22 [01:08<00:00,  3.13s/it]


generated embed with shape :(678, 384)
Adding 678 documents to VectorStore...
Successfully added 678 documents.
Total documents now: 5426


### Retriver Pipeline From VectorStore

In [15]:
import numpy as np
from typing import Dict, Any

class RAG_Retriever:
    """Retrieve similar documents from the Vector Store"""
    def __init__(self, vector_store, embedd_manager):
        """
        vector_store: instance of VectorStore
        embed_manager: instance of EmbedingManager
        """
        self.vector_store = vector_store
        self.embedd_manager = embedd_manager
        
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> list[Dict[str, Any]]:
        """Retrieve relevant documents for a query"""
        print(f"🔍 Retrieving relevant documents for query: '{query}'")
        print(f"top_k: {top_k}, score_threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedd_manager.generate_embedding([query])[0]
        
        try:
            # Search in vector store
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            retrieved_docs = []
            
            if results["documents"] and results["documents"][0]:
                for i, (document, metadata, distance, doc_id) in enumerate(
                    zip(results["documents"][0], results["metadatas"][0], results["distances"][0], results["ids"][0])
                ):
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} relevant documents.")
            else:
                print("No documents found.")
            
            return retrieved_docs
        
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


In [16]:
rag_retrieval = RAG_Retriever(vecstore, Embedd_manger)
rag_retrieval.retrieve("WHAT ARE RAG")


🔍 Retrieving relevant documents for query: 'WHAT ARE RAG'
top_k: 5, score_threshold: 0.0
generated embedings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.14it/s]


generated embed with shape :(1, 384)
Retrieved 5 relevant documents.


[{'id': 'doc_71392ace_0',
  'content': 'Mastering\nRAG\nA comprehensive guide for building\nenterprise-grade RAG systems',
  'metadata': {'keywords': '',
   'format': 'PDF 1.7',
   'total_pages': 194,
   'author': '',
   'page': 0,
   'file_path': '..\\data\\pdf\\Mastering RAG.pdf',
   'producer': 'Pdftools SDK',
   'creator': '',
   'content_len': 77,
   'creationDate': '',
   'creationdate': '',
   'moddate': '2024-09-27T00:48:54+00:00',
   'source': '..\\data\\pdf\\Mastering RAG.pdf',
   'subject': '',
   'title': '',
   'trapped': '',
   'doc_index': 0,
   'modDate': 'D:20240927004854Z'},
  'similarity_score': 0.292955219745636,
  'distance': 0.707044780254364,
  'rank': 1},
 {'id': 'doc_b8b69e2c_0',
  'content': 'Mastering\nRAG\nA comprehensive guide for building\nenterprise-grade RAG systems',
  'metadata': {'producer': 'Pdftools SDK',
   'doc_index': 0,
   'content_len': 77,
   'subject': '',
   'source': '..\\data\\pdf\\Mastering RAG.pdf',
   'file_path': '..\\data\\pdf\\Master

### Integration Vectordb Context pipeline With LLM output

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

# Initialize the Groq LLM
groq_api_key = "  "
llm=ChatGroq(groq_api_key=groq_api_key,model_name="moonshotai/kimi-k2-instruct",temperature=0.1,max_tokens=1024)

# 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    # retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    # generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content



In [24]:
answer=rag_simple("What is chunking?",rag_retrieval,llm)
print(answer)

🔍 Retrieving relevant documents for query: 'What is chunking?'
top_k: 3, score_threshold: 0.0
generated embedings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.34it/s]

generated embed with shape :(1, 384)
Retrieved 3 relevant documents.


Chunking is the process of splitting data into smaller, size-limited pieces (chunks) while handling oversized ones by issuing warnings.


### Enhanced RAG Pipeline Features
#### However, this version:

- Measures the level of confidence in the answer.
- Provides the sources from which the information was taken.
- You can also access the full context if you wish.

In [25]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    # The highest similarity score is chosen as a measure of "confidence" in the result (the higher the score, the more accurate the answer).
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is chunking?", rag_retrieval, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

🔍 Retrieving relevant documents for query: 'What is chunking?'
top_k: 3, score_threshold: 0.1
generated embedings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.32it/s]

generated embed with shape :(1, 384)
Retrieved 3 relevant documents.


Answer: Chunking is the process of splitting text into smaller pieces (“chunks”) that stay within a specified size limit; if any chunk exceeds that limit, the method issues a warning.
Sources: [{'source': '..\\data\\pdf\\Mastering RAG.pdf', 'page': 44, 'score': 0.3035711646080017, 'preview': 'method ensures that the chunks are within the speciﬁed size limits and handles edge cases,\nsuch as chunks longer than the speciﬁed size by issuing a warning\n*-)\nwww.rungalileo.io...'}, {'source': '..\\data\\pdf\\Mastering RAG.pdf', 'page': 44, 'score': 0.3035711646080017, 'preview': 'method ensures that the chunks are within the speciﬁed size limits and handles edge cases,\nsuch as chunks longer than the speciﬁed size by issuing a warning\n*-)\nwww.rungalileo.io...'}, {'source': '..\\data\\pdf\\Mastering RAG.pdf', 'page': 44, 'score': 0.3035711646080017, 'preview': 'method ensures that the chunks are within the speciﬁed size limits and handles edge cases,\nsuch as chunks longer than the speciﬁe

### More Enhanced RAG Pipeline Features
#### However, this version:
- Streams the answer gradually if enabled (simulates real-time generation).
- Optionally summarizes the generated answer.
- Keeps full query history for tracking or reuse.

In [26]:
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retrieval, llm)
result = adv_rag.query("What is chunking?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

🔍 Retrieving relevant documents for query: 'What is chunking?'
top_k: 3, score_threshold: 0.1
generated embedings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.00it/s]

generated embed with shape :(1, 384)
Retrieved 3 relevant documents.
Streaming answer:
Use the following context to answer the question concisely.
Context:
method ensures that the chunks are within the speciﬁed size limits and handles edge cases,


such as chunks longer than the speciﬁed size by issuing a warning
*-)
www.rungalileo.io

method ensures that the chunks are within the speciﬁed size limits and handles edge cases,
such as chunks longer than the speciﬁed size by issuing a warning
*-)
www.rungalileo.io

method ensures that the chunks are within the speciﬁed size limits and handles edge cases,
such as chunks longer than the speciﬁed size by issuing a warning
*-)
www.rungalileo.io

Question: What is chunking?

Answer:

Final Answer: Chunking is the process of splitting content into smaller pieces (“chunks”) that stay within a specified size limit; if a chunk exceeds the limit, the method warns about the oversized segment.

Citations:
[1] ..\data\pdf\Mastering RAG.pdf (page 44)
[2] ..\data\pdf\Mastering RAG.pdf (page 44)
[3] ..\data\pdf\Mastering RAG.pdf (page 44)
Summary: Chunking breaks content into smaller pieces under a size limit, flagging any segments that exceed it.
History: {'question': 'What is chunking?', 'answer'